In [2]:
import sqlite3
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

In [3]:
DATA_DIR = Path("fresh_inventory_data")
DATA_DIR.mkdir(exist_ok=True)

WAREHOUSE_A_DB = DATA_DIR / "warehouse_a.db"
WAREHOUSE_B_DB = DATA_DIR / "warehouse_b.db"
LEDGER_DB = DATA_DIR / "central_ledger.db"

print("Fresh database folder is ready.")

Fresh database folder is ready.


In [4]:
def create_warehouse_a():
    with sqlite3.connect(WAREHOUSE_A_DB) as connection:
        connection.execute("""
        CREATE TABLE IF NOT EXISTS movements (
            source_sequence INTEGER PRIMARY KEY,
            source_movement_id TEXT NOT NULL UNIQUE,
            order_reference TEXT NOT NULL,
            sku TEXT NOT NULL,
            movement_type TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            occurred_at TEXT NOT NULL,
            correction_of TEXT
        )
        """)

        movements = [
            (1, "A-001", "ORDER-1001", "SKU-CHAIR", "sale", -5, "2026-08-17T09:00:00Z", None),
            (2, "A-002", "ORDER-1002", "SKU-DESK", "sale", -2, "2026-08-17T09:10:00Z", None),
        ]

        connection.executemany("""
        INSERT OR IGNORE INTO movements (
            source_sequence,
            source_movement_id,
            order_reference,
            sku,
            movement_type,
            quantity,
            occurred_at,
            correction_of
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, movements)

    print("Warehouse A created with 2 ordered movements.")

create_warehouse_a()

Warehouse A created with 2 ordered movements.


In [5]:
def create_warehouse_b():
    with sqlite3.connect(WAREHOUSE_B_DB) as connection:
        connection.execute("""
        CREATE TABLE IF NOT EXISTS movements (
            source_sequence INTEGER PRIMARY KEY,
            source_movement_id TEXT NOT NULL UNIQUE,
            order_reference TEXT NOT NULL,
            sku TEXT NOT NULL,
            movement_type TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            occurred_at TEXT NOT NULL,
            correction_of TEXT
        )
        """)

        movements = [
            (1, "B-001", "ORDER-1001", "SKU-CHAIR", "sale", -5, "2026-08-17T09:00:05Z", None),
            (2, "B-002", "RESTOCK-2001", "SKU-LAMP", "restock", 20, "2026-08-17T09:15:00Z", None),
            (3, "B-003", "ORDER-9999", "SKU-CHAIR", "sale", -3, "2026-08-17T09:20:00Z", None),
            (4, "B-004", "ORDER-9999", "SKU-CHAIR", "sale_correction", 3, "2026-08-17T09:25:00Z", "B-003"),
        ]

        connection.executemany("""
        INSERT OR IGNORE INTO movements (
            source_sequence,
            source_movement_id,
            order_reference,
            sku,
            movement_type,
            quantity,
            occurred_at,
            correction_of
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, movements)

    print("Warehouse B created with 4 ordered movements.")

create_warehouse_b()

Warehouse B created with 4 ordered movements.


In [6]:
def create_central_ledger():
    with sqlite3.connect(LEDGER_DB) as connection:
        connection.execute("""
        CREATE TABLE IF NOT EXISTS ledger_movements (
            ledger_movement_id INTEGER PRIMARY KEY AUTOINCREMENT,
            idempotency_key TEXT NOT NULL UNIQUE,
            order_reference TEXT NOT NULL,
            sku TEXT NOT NULL,
            movement_type TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            occurred_at TEXT NOT NULL,
            first_seen_from TEXT NOT NULL,
            source_movement_ids TEXT NOT NULL,
            synced_at TEXT NOT NULL
        )
        """)

        connection.execute("""
        CREATE TABLE IF NOT EXISTS sync_watermarks (
            warehouse_name TEXT PRIMARY KEY,
            last_source_sequence INTEGER NOT NULL DEFAULT 0,
            updated_at TEXT NOT NULL
        )
        """)

        connection.executemany("""
        INSERT OR IGNORE INTO sync_watermarks (
            warehouse_name,
            last_source_sequence,
            updated_at
        )
        VALUES (?, 0, ?)
        """, [
            ("Warehouse A", datetime.now(timezone.utc).isoformat()),
            ("Warehouse B", datetime.now(timezone.utc).isoformat()),
        ])

    print("Central ledger and warehouse watermarks created.")

create_central_ledger()

Central ledger and warehouse watermarks created.


In [7]:
def get_watermark(warehouse_name):
    with sqlite3.connect(LEDGER_DB) as connection:
        row = connection.execute("""
        SELECT last_source_sequence
        FROM sync_watermarks
        WHERE warehouse_name = ?
        """, (warehouse_name,)).fetchone()

    return row[0]


def read_new_warehouse_movements(database_path, warehouse_name):
    watermark = get_watermark(warehouse_name)

    with sqlite3.connect(database_path) as connection:
        connection.row_factory = sqlite3.Row

        rows = connection.execute("""
        SELECT
            source_sequence,
            source_movement_id,
            order_reference,
            sku,
            movement_type,
            quantity,
            occurred_at,
            correction_of
        FROM movements
        WHERE source_sequence > ?
        ORDER BY source_sequence
        """, (watermark,)).fetchall()

    return [dict(row) | {"warehouse": warehouse_name} for row in rows]

In [8]:
def build_idempotency_key(movement):
    if movement["movement_type"] == "sale_correction":
        raw_key = "|".join([
            "correction",
            movement["correction_of"],
            movement["sku"],
            str(movement["quantity"]),
        ])
    else:
        raw_key = "|".join([
            movement["movement_type"],
            movement["order_reference"],
            movement["sku"],
            str(movement["quantity"]),
        ])

    return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()

In [9]:
def sync_inventory():
    sources = [
        (WAREHOUSE_A_DB, "Warehouse A"),
        (WAREHOUSE_B_DB, "Warehouse B"),
    ]

    applied_count = 0
    duplicate_count = 0

    with sqlite3.connect(LEDGER_DB) as ledger_connection:
        for database_path, warehouse_name in sources:
            new_movements = read_new_warehouse_movements(
                database_path,
                warehouse_name
            )

            if not new_movements:
                print(f"{warehouse_name}: no new movements.")
                continue

            for movement in new_movements:
                idempotency_key = build_idempotency_key(movement)

                cursor = ledger_connection.execute("""
                INSERT INTO ledger_movements (
                    idempotency_key,
                    order_reference,
                    sku,
                    movement_type,
                    quantity,
                    occurred_at,
                    first_seen_from,
                    source_movement_ids,
                    synced_at
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(idempotency_key) DO NOTHING
                """, (
                    idempotency_key,
                    movement["order_reference"],
                    movement["sku"],
                    movement["movement_type"],
                    movement["quantity"],
                    movement["occurred_at"],
                    warehouse_name,
                    json.dumps([movement["source_movement_id"]]),
                    datetime.now(timezone.utc).isoformat(),
                ))

                if cursor.rowcount == 1:
                    applied_count += 1
                    print(
                        f"APPLIED: {warehouse_name} "
                        f"{movement['source_movement_id']}"
                    )
                else:
                    duplicate_count += 1
                    print(
                        f"DUPLICATE SKIPPED: {warehouse_name} "
                        f"{movement['source_movement_id']}"
                    )

            highest_sequence = new_movements[-1]["source_sequence"]

            ledger_connection.execute("""
            UPDATE sync_watermarks
            SET last_source_sequence = ?,
                updated_at = ?
            WHERE warehouse_name = ?
            """, (
                highest_sequence,
                datetime.now(timezone.utc).isoformat(),
                warehouse_name,
            ))

    print(
        f"SYNC COMPLETE: {applied_count} applied, "
        f"{duplicate_count} duplicates skipped."
    )

In [10]:
sync_inventory()

APPLIED: Warehouse A A-001
APPLIED: Warehouse A A-002
DUPLICATE SKIPPED: Warehouse B B-001
APPLIED: Warehouse B B-002
APPLIED: Warehouse B B-003
APPLIED: Warehouse B B-004
SYNC COMPLETE: 5 applied, 1 duplicates skipped.


In [11]:
with sqlite3.connect(LEDGER_DB) as connection:
    rows = connection.execute("""
    SELECT
        ledger_movement_id,
        order_reference,
        sku,
        movement_type,
        quantity,
        first_seen_from
    FROM ledger_movements
    ORDER BY ledger_movement_id
    """).fetchall()

print(f"Central ledger contains {len(rows)} movements.")
for row in rows:
    print(row)

Central ledger contains 5 movements.
(1, 'ORDER-1001', 'SKU-CHAIR', 'sale', -5, 'Warehouse A')
(2, 'ORDER-1002', 'SKU-DESK', 'sale', -2, 'Warehouse A')
(4, 'RESTOCK-2001', 'SKU-LAMP', 'restock', 20, 'Warehouse B')
(5, 'ORDER-9999', 'SKU-CHAIR', 'sale', -3, 'Warehouse B')
(6, 'ORDER-9999', 'SKU-CHAIR', 'sale_correction', 3, 'Warehouse B')


In [12]:
sync_inventory()

with sqlite3.connect(LEDGER_DB) as connection:
    ledger_count = connection.execute("""
    SELECT COUNT(*)
    FROM ledger_movements
    """).fetchone()[0]

assert ledger_count == 5, (
    f"Expected 5 ledger movements after a second sync, "
    f"but found {ledger_count}."
)

print("TEST PASSED: Second sync created no duplicate movements.")

Warehouse A: no new movements.
Warehouse B: no new movements.
SYNC COMPLETE: 0 applied, 0 duplicates skipped.
TEST PASSED: Second sync created no duplicate movements.


In [13]:
with sqlite3.connect(LEDGER_DB) as connection:
    stock_summary = connection.execute("""
    SELECT
        sku,
        SUM(quantity) AS net_stock_change
    FROM ledger_movements
    GROUP BY sku
    ORDER BY sku
    """).fetchall()

print("Net stock change by SKU:")
for sku, net_stock_change in stock_summary:
    print(f"{sku}: {net_stock_change}")

Net stock change by SKU:
SKU-CHAIR: -5
SKU-DESK: -2
SKU-LAMP: 20
